# Images and animation 🖼️

Any `image.Image` is drawn right in the cell's output, with colored half
blocks (two pixels per character). End a cell with an image, or pass it to
`nb.Display`. `nb.DisplayID` replaces its previous output, which is all an
animation needs.

Press `A` to run every cell.

In [ ]:
import (
	"image"
	"image/color"
	"image/draw"
	"math"
	"math/rand/v2"
)

// Mandelbrot renders the Mandelbrot set around (cx, cy); scale is the width
// of the view in the complex plane.
func Mandelbrot(w, h int, cx, cy, scale float64, maxIter int) image.Image {
	img := image.NewRGBA(image.Rect(0, 0, w, h))
	for py := range h {
		for px := range w {
			x := cx + (float64(px)/float64(w)-0.5)*scale
			y := cy + (float64(py)/float64(h)-0.5)*scale*float64(h)/float64(w)
			img.Set(px, py, escapeColor(x, y, maxIter))
		}
	}
	return img
}

// escapeColor colors a point by how fast it escapes, with a smooth
// iteration count mapped onto a cosine palette.
func escapeColor(x, y float64, maxIter int) color.Color {
	var zx, zy float64
	for n := range maxIter {
		zx, zy = zx*zx-zy*zy+x, 2*zx*zy+y
		if r2 := zx*zx + zy*zy; r2 > 16 {
			t := (float64(n) + 1 - math.Log2(math.Log(r2)/2)) / 48
			return color.RGBA{wave(t, 0), wave(t, 0.1), wave(t, 0.2), 255}
		}
	}
	return color.Black
}

func wave(t, phase float64) uint8 {
	return uint8(127.5 + 127.5*math.Cos(2*math.Pi*(t+phase)))
}

A trailing expression is displayed, images included:

In [ ]:
Mandelbrot(640, 400, -0.6, 0, 3.4, 200)

## Animation

Each `nb.DisplayID("zoom", ...)` replaces the frame before it. This zooms into
the *Seahorse Valley*:

In [ ]:
for i := range 36 {
	scale := 3.0 * math.Pow(0.82, float64(i))
	nb.DisplayID("zoom", Mandelbrot(240, 150, -0.7436447860, 0.1318252536, scale, 80+12*i))
}

## Game of Life

A `Life` value declared with `:=` is kept for later cells (it's saved with
`encoding/gob` when the cell ends), so the next cell carries on where this
one stopped.

In [ ]:
// Life is Conway's Game of Life on a torus. Age counts how long each cell
// has been alive (0 for dead cells).
type Life struct {
	W, H int
	Age  []uint8
}

func NewLife(w, h int, seed uint64) *Life {
	r := rand.New(rand.NewPCG(seed, 1))
	l := &Life{W: w, H: h, Age: make([]uint8, w*h)}
	for i := range l.Age {
		if r.IntN(4) == 0 {
			l.Age[i] = 1
		}
	}
	return l
}

func (l *Life) alive(x, y int) bool {
	return l.Age[(y+l.H)%l.H*l.W+(x+l.W)%l.W] > 0
}

// Step advances one generation.
func (l *Life) Step() {
	next := make([]uint8, len(l.Age))
	for y := range l.H {
		for x := range l.W {
			n := 0
			for dy := -1; dy <= 1; dy++ {
				for dx := -1; dx <= 1; dx++ {
					if (dx != 0 || dy != 0) && l.alive(x+dx, y+dy) {
						n++
					}
				}
			}
			i := y*l.W + x
			switch {
			case l.Age[i] > 0 && (n == 2 || n == 3):
				next[i] = min(l.Age[i]+1, 255)
			case l.Age[i] == 0 && n == 3:
				next[i] = 1
			}
		}
	}
	l.Age = next
}

// Image draws newborn cells in yellow, fading to blue as they age.
func (l *Life) Image() image.Image {
	img := image.NewRGBA(image.Rect(0, 0, l.W, l.H))
	draw.Draw(img, img.Bounds(), image.NewUniform(color.RGBA{16, 18, 28, 255}), image.Point{}, draw.Src)
	for i, age := range l.Age {
		if age == 0 {
			continue
		}
		t := min(float64(age)/12, 1)
		img.Set(i%l.W, i/l.W, color.RGBA{uint8(255 - 200*t), uint8(220 - 50*t), uint8(80 + 175*t), 255})
	}
	return img
}

life := NewLife(80, 48, 42)
for range 60 {
	nb.DisplayID("life", life.Image())
	life.Step()
	time.Sleep(30 * time.Millisecond)
}

In [ ]:
// life is still here: 60 more generations.
for range 60 {
	nb.DisplayID("life", life.Image())
	life.Step()
	time.Sleep(30 * time.Millisecond)
}

## Plotting

There's no plotting library here: drawing into an `image.RGBA` is enough for
a quick chart.

In [ ]:
// Plot draws functions over [x0, x1], scaling y to fit.
func Plot(w, h int, x0, x1 float64, fns ...func(float64) float64) image.Image {
	img := image.NewRGBA(image.Rect(0, 0, w, h))
	draw.Draw(img, img.Bounds(), image.NewUniform(color.RGBA{24, 24, 32, 255}), image.Point{}, draw.Src)
	ys := make([][]float64, len(fns))
	lo, hi := math.Inf(1), math.Inf(-1)
	for i, f := range fns {
		ys[i] = make([]float64, w)
		for px := range w {
			y := f(x0 + (x1-x0)*float64(px)/float64(w-1))
			ys[i][px] = y
			lo, hi = min(lo, y), max(hi, y)
		}
	}
	if hi == lo {
		hi++
	}
	row := func(y float64) int { return int(math.Round((hi - y) / (hi - lo) * float64(h-1))) }
	if lo < 0 && hi > 0 {
		for px := range w {
			img.Set(px, row(0), color.RGBA{70, 70, 90, 255})
		}
	}
	colors := []color.RGBA{{0, 173, 216, 255}, {255, 180, 60, 255}, {230, 90, 120, 255}, {120, 220, 120, 255}}
	for i, samples := range ys {
		for px := 1; px < w; px++ {
			// Join each sample to the previous one with a vertical run.
			a, b := row(samples[px-1]), row(samples[px])
			for py := min(a, b); py <= max(a, b); py++ {
				img.Set(px, py, colors[i%len(colors)])
			}
		}
	}
	return img
}

Plot(240, 90, -2*math.Pi, 2*math.Pi,
	math.Sin,
	math.Cos,
	func(x float64) float64 { return math.Sin(3*x) * math.Exp(-x*x/10) },
)